# PathSnap — `features.py` notebook

This notebook is the executable companion to `features.py`. It builds one row per **GPS observation × candidate road segment**, then audits the feature matrix that will be shared by the requested model families.

The supplied PathSnap dataset document says `.track` contains longitude, latitude and timestamp; `.nodes` contains road-network node coordinates; `.arcs` contains endpoint node IDs; and `.route` contains a chronological sequence of reference arc IDs. It also explicitly warns not to assume a GPS row corresponds directly to a road-network row.

The reference-route-to-observation association is therefore intentionally kept outside this feature extractor.

In [ ]:
# Install once if necessary
# %pip install numpy pandas scikit-learn lightgbm xgboost joblib

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# If features.py is beside this notebook, use:
# sys.path.insert(0, str(Path('.').resolve()))

from features import FeatureConfig, build_dataset, model_feature_columns, save_features

In [ ]:
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'candidate_features.csv'

CONFIG = FeatureConfig(
    n_candidates=8,
    max_candidate_distance_m=250.0,
)

print('Dataset:', DATASET_ROOT)
print('Output:', OUTPUT_PATH)
print(CONFIG)

In [ ]:
# Inspect available trips before feature extraction
trip_dirs = sorted(p for p in DATASET_ROOT.iterdir() if p.is_dir())
print('Trips found:', len(trip_dirs))
for trip in trip_dirs[:10]:
    print('\n', trip.name)
    for f in sorted(trip.iterdir()):
        print('  ', f.name)

In [ ]:
# IMPORTANT: for the training matrix, pass ONLY the 80 training trips.
# Do not mix evaluator-testing trips into feature fitting/selection.

features = build_dataset(
    DATASET_ROOT,
    trip_dirs=trip_dirs,  # replace with your 80 training trips after partition inspection
    config=CONFIG,
)

print('Shape:', features.shape)
display(features.head())

In [ ]:
print('Trips:', features.trip_id.nunique())
print('Observations:', features.obs_index.nunique())
print('Candidate arcs:', features.arc_id.nunique())

print('\nMissing-value fraction:')
display(features.isna().mean().sort_values(ascending=False).to_frame('missing_fraction'))

print('\nNumeric summary:')
display(features.select_dtypes(include=np.number).describe().T)

In [ ]:
# Candidate coverage — this is an important diagnostic before model training.
coverage = features.groupby(['trip_id','obs_index']).size()
print('Candidate count per observation:')
display(coverage.describe().to_frame('candidate_count'))

print('Observations represented in candidate table:', coverage.size)

In [ ]:
FEATURE_COLUMNS = model_feature_columns(features)
print(f'{len(FEATURE_COLUMNS)} numeric model features:')
for c in FEATURE_COLUMNS:
    print(' -', c)

X = features[FEATURE_COLUMNS]
metadata = features[['trip_id','obs_index','arc_id','candidate_rank']]
print('\nX shape:', X.shape)
print('Metadata shape:', metadata.shape)

In [ ]:
save_features(features, OUTPUT_PATH)
manifest = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'dtype': [str(features[c].dtype) for c in FEATURE_COLUMNS],
    'missing_fraction': [features[c].isna().mean() for c in FEATURE_COLUMNS],
})
manifest_path = OUTPUT_PATH.with_name('feature_manifest.csv')
manifest.to_csv(manifest_path, index=False)
print('Saved:', OUTPUT_PATH)
print('Saved:', manifest_path)

## Target-label step — keep this separate

Do **not** label rows by assuming `obs_index == route_position`. The supplied dataset documentation warns that GPS observations and road-network rows are not directly aligned.

After your team validates how the reference route should be associated with GPS observations, create `reference_arc_id` per observation and then define the supervised candidate target, for example:

```python
features['target'] = (features['arc_id'] == features['reference_arc_id']).astype(int)
```

For the requested classifiers, this gives a binary candidate-ranking problem. Keep `arc_id`, `trip_id`, and `obs_index` as metadata rather than predictive features.

For Logistic Regression, SGD/Ridge, KNN, MLP and optional linear SVM, fit scaling on training data only. Tree/boosting models do not require standardisation.